In [0]:
import logging
import requests
from pyspark.sql.functions import lit, current_timestamp

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

BASE_URL = "https://api.escuelajs.co/api/v1"

def extract_bronze(endpoint, table_name):
    logger.info(f"Starting extraction — table: {table_name}, endpoint: {BASE_URL}/{endpoint}")
    try:
        response = requests.get(f"{BASE_URL}/{endpoint}")
        response.raise_for_status()
        data = response.json()

        if not data:
            raise Exception(f"API returned empty response for endpoint: {endpoint}")

        logger.info(f"API response received — {len(data)} records for {table_name}")

        flattened = []
        for record in data:
            flat = {}
            for key, value in record.items():
                if isinstance(value, dict):
                    for subkey, subvalue in value.items():
                        flat[f"{key}_{subkey}"] = str(subvalue)
                elif isinstance(value, list):
                    flat[key] = ",".join([str(i) for i in value])  # "https://pic1.jpg,https://pic2.jpg"
                else:
                    flat[key] = value
            flattened.append(flat)

        logger.info(f"Data flattened — {len(flattened)} records")

        df = spark.createDataFrame(flattened)
        logger.info(f"DataFrame created — {df.count()} rows, {len(df.columns)} columns")

        df = df.withColumn("source", lit("platzi_api")) \
               .withColumn("ingestion_time", current_timestamp())

        df.write.mode("overwrite") \
            .format("delta") \
            .saveAsTable(f"workspace.ecommerce_bronze.{table_name}")

        logger.info(f"Successfully written {table_name} to Bronze — {df.count()} rows")

    except requests.exceptions.RequestException as e:
        logger.error(f"Request failed for {endpoint}: {e}")
        raise

    except Exception as e:
        logger.error(f"Unexpected error in {table_name}: {e}")
        raise

extract_bronze("products", "products")
extract_bronze("users", "users")
extract_bronze("categories", "categories")